In [118]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text
# Root проекта
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
root = next((p for p in candidates if (p / "pyproject.toml").exists()), Path.cwd())
if (root / "research").is_dir() and str(root) not in sys.path:
    sys.path.insert(0, str(root))

from research.db.connection import get_engine

data_dir = root / "data" / "reconstructed"

In [110]:
# Путь к Parquet (подставь свой файл)
# data_dir из первой ячейки (root / data / reconstructed) — выполни её

# Путь к одному часовому parquet
parquet_path = data_dir / "BTC-USDT-SWAP" / "grid100ms" / "2026-03-22" / (
    "book_grid100ms_BTC-USDT-SWAP_2026-03-23_01-00-00__2026-03-23_02-00-00.parquet"
)
if not parquet_path.exists():
    found = sorted(data_dir.rglob("*.parquet"))
    parquet_path = found[0] if found else parquet_path
    if found:
        print("Файл по умолчанию не найден, загружаю:", parquet_path)

df = pd.read_parquet(parquet_path)
df

Файл по умолчанию не найден, загружаю: d:\tumar\okx-hft-research\data\reconstructed\BTC-USDT-SWAP\grid100ms\2026-03-23\book_grid100ms_BTC-USDT-SWAP_2026-03-23_00-00-00__2026-03-23_01-00-00.parquet


,ts_event,inst_id,anchor_snapshot_ts,reconstruction_mode,mid_px,spread_px,bid_px_01,bid_sz_01,bid_px_02,bid_sz_02,...,ask_px_06,ask_sz_06,ask_px_07,ask_sz_07,ask_px_08,ask_sz_08,ask_px_09,ask_sz_09,ask_px_10,ask_sz_10
0,2026-03-23 00:00:00+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,67830.8,456.44,67830.7,0.43,...,67831.6,0.75,67831.7,13.78,67831.8,0.02,67831.9,70.47,67832.1,0.05
1,2026-03-23 00:00:00.100000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,67830.8,466.58,67830.7,0.43,...,67831.8,0.03,67831.9,70.47,67832.1,0.05,67832.2,0.06,67832.3,22.34
2,2026-03-23 00:00:00.200000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,67830.8,488.05,67830.7,0.43,...,67831.9,0.05,67832.1,0.05,67832.2,0.05,67832.3,0.06,67832.4,0.03
3,2026-03-23 00:00:00.300000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67840.65,0.1,67840.6,1032.91,67840.5,8.02,...,67842.1,0.03,67842.2,0.01,67842.7,0.04,67842.8,0.90,67843.1,0.01
4,2026-03-23 00:00:00.400000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67843.85,0.1,67843.8,914.75,67843.7,0.01,...,67844.6,0.01,67845.1,0.03,67845.3,0.01,67845.4,0.33,67845.6,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35996,2026-03-23 00:59:59.600000+00:00,BTC-USDT-SWAP,2026-03-23 00:59:47.708000+00:00,grid_100ms,67640.55,0.1,67640.5,406.80,67640.4,0.01,...,67641.2,0.05,67641.3,0.05,67641.4,0.03,67641.5,0.05,67641.6,0.06
35997,2026-03-23 00:59:59.700000+00:00,BTC-USDT-SWAP,2026-03-23 00:59:47.708000+00:00,grid_100ms,67640.55,0.1,67640.5,408.99,67640.4,0.01,...,67641.2,0.05,67641.3,0.05,67641.4,0.03,67641.5,0.05,67641.6,0.06
35998,2026-03-23 00:59:59.800000+00:00,BTC-USDT-SWAP,2026-03-23 00:59:47.708000+00:00,grid_100ms,67640.55,0.1,67640.5,397.49,67640.4,0.01,...,67641.2,0.05,67641.3,0.05,67641.4,0.03,67641.5,0.05,67641.6,0.06
35999,2026-03-23 00:59:59.900000+00:00,BTC-USDT-SWAP,2026-03-23 00:59:47.708000+00:00,grid_100ms,67640.55,0.1,67640.5,397.49,67640.4,0.01,...,67641.2,0.05,67641.3,0.05,67641.4,0.03,67641.5,0.05,67641.6,0.06


## Canonical validation (snapshot-aligned)

Ниже — корректный pipeline оценки качества восстановления стакана:

- **snapshot_aligned** — основной и единственно корректный verdict.
- **grid_nearest** — только baseline для контраста (показывает ложные mismatch из-за несовпадения времени grid и snapshot).

> Важно: если результаты старого блока выше отличаются, доверять нужно именно этому блоку.

In [120]:
from pathlib import Path
import sys
import json
import subprocess
import pandas as pd

# =============================
# CONFIG
# =============================
INST_ID = "BTC-USDT-SWAP"
START = "2026-03-23 01:00:00"
END = "2026-03-23 02:00:00"
DEPTH = 10
WORST_K = 20
SNAPSHOT_LIMIT = 1000  # 0 = no limit
RUN_GRID_BASELINE = True

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
]
root = next((p for p in candidates if (p / "pyproject.toml").exists()), Path.cwd())

script_path = root / "scripts" / "validate_orderbook_reconstruction.py"
out_root = root / "data" / "diagnostics" / "orderbook_validation"
out_root.mkdir(parents=True, exist_ok=True)

if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path}")

# =============================
# 1) SNAPSHOT-ALIGNED (correct)
# =============================
cmd_snapshot = [
    sys.executable,
    str(script_path),
    "--mode", "snapshot_aligned",
    "--inst-id", INST_ID,
    "--schema", "okx_raw",
    "--start", START,
    "--end", END,
    "--depth", str(DEPTH),
    "--worst-k", str(WORST_K),
    "--snapshot-limit", str(SNAPSHOT_LIMIT),
    "--output-dir", str(out_root),
]

print("RUN:", " ".join(cmd_snapshot))
subprocess.run(cmd_snapshot, check=True)

snapshot_dir = out_root / "snapshot_aligned" / f"{INST_ID}_{pd.Timestamp(START).strftime('%Y-%m-%d_%H-%M-%S')}__{pd.Timestamp(END).strftime('%Y-%m-%d_%H-%M-%S')}"
snapshot_summary_path = snapshot_dir / "validation_summary.json"

if not snapshot_summary_path.exists():
    raise FileNotFoundError(f"Snapshot summary not found: {snapshot_summary_path}")

snapshot_summary = json.loads(snapshot_summary_path.read_text(encoding="utf-8"))

# =============================
# 2) GRID_NEAREST baseline
# =============================
grid_summary = None
if RUN_GRID_BASELINE:
    start_ts = pd.Timestamp(START, tz="UTC")
    end_ts = pd.Timestamp(END, tz="UTC")
    if end_ts <= start_ts:
        raise ValueError("END must be greater than START")

    mode_dir = root / "data" / "reconstructed" / INST_ID / "grid100ms"
    if not mode_dir.exists():
        raise FileNotFoundError(f"Reconstructed grid folder not found: {mode_dir}")

    # Collect all candidate parquet files and keep only those that overlap [START, END].
    # Filename format:
    # book_grid100ms_<INST>_YYYY-MM-DD_HH-MM-SS__YYYY-MM-DD_HH-MM-SS.parquet
    candidates = sorted(mode_dir.rglob(f"book_grid100ms_{INST_ID}_*.parquet"))
    if not candidates:
        raise FileNotFoundError(f"No reconstructed grid parquet files found under: {mode_dir}")

    parquet_paths = []
    for p in candidates:
        stem = p.stem
        try:
            # split only at the last 2 date-time chunks
            # e.g. book_grid100ms_BTC-USDT-SWAP_2026-03-22_00-00-00__2026-03-22_01-00-00
            left, right = stem.split("__")
            left_parts = left.rsplit("_", 2)
            start_str = f"{left_parts[-2]} {left_parts[-1].replace('-', ':')}"
            right_parts = right.rsplit("_", 1)
            end_str = f"{right_parts[0]} {right_parts[1].replace('-', ':')}"
            file_start = pd.Timestamp(start_str, tz="UTC")
            file_end = pd.Timestamp(end_str, tz="UTC")
        except Exception:
            continue

        # overlap check with [start_ts, end_ts]
        if file_end > start_ts and file_start < end_ts:
            parquet_paths.append((file_start, p))

    if not parquet_paths:
        raise FileNotFoundError(
            f"No overlapping grid parquet files found for window [{start_ts} .. {end_ts}] under {mode_dir}"
        )

    parquet_paths = [p for _, p in sorted(parquet_paths, key=lambda x: x[0])]

    merged_path = out_root / f"merged_tmp_{start_ts.strftime('%Y%m%d_%H%M%S')}__{end_ts.strftime('%Y%m%d_%H%M%S')}.parquet"
    merged_df = pd.concat([pd.read_parquet(p) for p in parquet_paths], ignore_index=True)
    merged_df["ts_event"] = pd.to_datetime(merged_df["ts_event"], utc=True, errors="coerce")
    merged_df = merged_df.sort_values("ts_event")
    merged_df = merged_df[(merged_df["ts_event"] >= start_ts) & (merged_df["ts_event"] <= end_ts)].copy()
    merged_df.to_parquet(merged_path, index=False)

    print(f"Merged {len(parquet_paths)} parquet files for baseline")

    cmd_grid = [
        sys.executable,
        str(script_path),
        "--mode", "grid_nearest",
        "--inst-id", INST_ID,
        "--schema", "okx_raw",
        "--snapshot-table", "orderbook_snapshots",
        "--reconstructed-parquet", str(merged_path),
        "--start", START,
        "--end", END,
        "--depth", str(DEPTH),
        "--worst-k", str(WORST_K),
        "--snapshot-limit", str(SNAPSHOT_LIMIT),
        "--output-dir", str(out_root),
    ]

    print("RUN:", " ".join(cmd_grid))
    subprocess.run(cmd_grid, check=True)

    grid_dir = out_root / "grid_nearest" / f"{INST_ID}_{pd.Timestamp(START).strftime('%Y-%m-%d_%H-%M-%S')}__{pd.Timestamp(END).strftime('%Y-%m-%d_%H-%M-%S')}"
    grid_summary_path = grid_dir / "validation_summary.json"
    if not grid_summary_path.exists():
        raise FileNotFoundError(f"Grid summary not found: {grid_summary_path}")

    grid_summary = json.loads(grid_summary_path.read_text(encoding="utf-8"))

# =============================
# 3) REPORT
# =============================
metrics = [
    "validated_points",
    "top1_bid_match_rate",
    "top1_ask_match_rate",
    "mean_px_diff",
    "p95_px_diff",
    "max_px_diff",
    "mean_sz_diff",
    "p95_sz_diff",
    "max_sz_diff",
]

rows = []
for m in metrics:
    row = {
        "metric": m,
        "snapshot_aligned": snapshot_summary.get(m),
    }
    if grid_summary is not None:
        row["grid_nearest"] = grid_summary.get(m)
    rows.append(row)

report_df = pd.DataFrame(rows)
display(report_df)

print("\nSnapshot-aligned summary:")
print(snapshot_summary_path)

if grid_summary is not None:
    print("\nGrid-nearest summary:")
    print(grid_summary_path)

worst_snapshot_path = snapshot_dir / "worst_points.csv"
if worst_snapshot_path.exists():
    print("\nWorst points (snapshot_aligned):")
    display(pd.read_csv(worst_snapshot_path).head(10))


RUN: d:\tumar\okx-hft-research\.venv\Scripts\python.exe d:\tumar\okx-hft-research\scripts\validate_orderbook_reconstruction.py --mode snapshot_aligned --inst-id BTC-USDT-SWAP --schema okx_raw --start 2026-03-23 01:00:00 --end 2026-03-23 02:00:00 --depth 10 --worst-k 20 --snapshot-limit 1000 --output-dir d:\tumar\okx-hft-research\data\diagnostics\orderbook_validation


C:\Users\sorok\AppData\Local\Temp\ipykernel_22148\1273247780.py:71: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hour_starts = pd.date_range(start=start_ts, end=end_ts - pd.Timedelta(hours=1), freq="1H", tz="UTC")


RUN: d:\tumar\okx-hft-research\.venv\Scripts\python.exe d:\tumar\okx-hft-research\scripts\validate_orderbook_reconstruction.py --mode grid_nearest --inst-id BTC-USDT-SWAP --schema okx_raw --snapshot-table orderbook_snapshots --reconstructed-parquet d:\tumar\okx-hft-research\data\diagnostics\orderbook_validation\merged_tmp_20260323_010000__20260323_020000.parquet --start 2026-03-23 01:00:00 --end 2026-03-23 02:00:00 --depth 10 --worst-k 20 --snapshot-limit 1000 --output-dir d:\tumar\okx-hft-research\data\diagnostics\orderbook_validation


,metric,snapshot_aligned,grid_nearest
0,validated_points,117.0,117.000000
1,top1_bid_match_rate,1.0,1.000000
2,top1_ask_match_rate,1.0,1.000000
3,mean_px_diff,0.0,0.011282
4,p95_px_diff,0.0,0.000000
5,max_px_diff,0.0,1.100000
6,mean_sz_diff,0.0,1.215389
7,p95_sz_diff,0.0,0.010000
8,max_sz_diff,0.0,340.120000



Snapshot-aligned summary:
d:\tumar\okx-hft-research\data\diagnostics\orderbook_validation\snapshot_aligned\BTC-USDT-SWAP_2026-03-23_01-00-00__2026-03-23_02-00-00\validation_summary.json

Grid-nearest summary:
d:\tumar\okx-hft-research\data\diagnostics\orderbook_validation\grid_nearest\BTC-USDT-SWAP_2026-03-23_01-00-00__2026-03-23_02-00-00\validation_summary.json

Worst points (snapshot_aligned):


,snapshot_ms,snapshot_ts,top1_bid_match,top1_ask_match,crossed_book_real,crossed_book_rec,max_px_diff,max_sz_diff,mean_px_diff,mean_sz_diff
0,1774227618408,2026-03-23T01:00:18.408000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
1,1774227649208,2026-03-23T01:00:49.208000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
2,1774227680208,2026-03-23T01:01:20.208000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
3,1774227711008,2026-03-23T01:01:51.008000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
4,1774227741708,2026-03-23T01:02:21.708000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
5,1774227773008,2026-03-23T01:02:53.008000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
6,1774227803708,2026-03-23T01:03:23.708000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
7,1774227834308,2026-03-23T01:03:54.308000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
8,1774227865008,2026-03-23T01:04:25.008000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
9,1774227895608,2026-03-23T01:04:55.608000+00:00,True,True,False,False,0.0,0.0,0.0,0.0
